In [16]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import random
import time
import json
import os

from IPython.display import display


In [17]:
# Enable interactive matplotlib backend for Jupyter
# Run this cell first to enable click interactions
# If clicking doesn't work, try: pip install ipympl, then restart kernel
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    print("Using matplotlib widget backend (interactive)")
except:
    try:
        get_ipython().run_line_magic('matplotlib', 'notebook')
        print("Using matplotlib notebook backend (interactive)")
    except:
        get_ipython().run_line_magic('matplotlib', 'inline')
        print("Using inline backend (limited interactivity)")
        print("For full interactivity, install: pip install ipympl")


Using matplotlib notebook backend (interactive)


In [18]:
# Model class and loading (same as training notebook)
class MLPIntegrator(nn.Module):
    def __init__(self, k=5, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(k*3, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def load_integrator_model(model_path, device="cpu"):
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model = MLPIntegrator(k=checkpoint['k'], hidden=checkpoint['hidden']).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    config = {k: checkpoint[k] for k in ['k', 'R', 'hidden', 'u_scale', 'Nx', 'Nt']}
    print(f"Model loaded: k={config['k']}, R={config['R']}, u_scale={config['u_scale']:.6e}")
    return model, config


In [19]:
# Helper functions (reuse from File 1 or define here)
def build_bc_mask(Nt, Nx):
    is_bc = np.zeros((Nt, Nx), dtype=bool)
    is_bc[:, 0] = True
    is_bc[:, Nx-1] = True
    return is_bc

def candidates_from_interior_elliptical(visited, is_bc, R, Rt, Rx):
    """Generate candidates from elliptical neighborhoods (from File 1)."""
    Nt, Nx = visited.shape
    interior_centers = np.argwhere(visited & (~is_bc))
    cand = set()
    for (t0, x0) in interior_centers:
        t_min, t_max = max(0, int(t0 - Rt)), min(Nt-1, int(t0 + Rt))
        x_min, x_max = max(0, int(x0 - Rx)), min(Nx-1, int(x0 + Rx))
        for t in range(t_min, t_max+1):
            for x in range(x_min, x_max+1):
                dt, dx = (t - t0) / Rt, (x - x0) / Rx
                if not visited[t, x] and dt*dt + dx*dx <= 1.0:
                    cand.add((t, x))
    return list(cand), interior_centers

def nearest5_elliptical(visited_pts, sampled, Rt, Rx, k=5):
    """Select nearest-k using elliptical metric (from File 1)."""
    if len(visited_pts) < k:
        return visited_pts
    dt = (visited_pts[:, 0] - sampled[0]) / Rt
    dx = (visited_pts[:, 1] - sampled[1]) / Rx
    d = dt*dt + dx*dx
    return visited_pts[np.argsort(d)[:k]]


In [20]:
# -----------------------
# 2. Feature construction for model inference
# -----------------------

def build_features_for_model(visited_pts, sampled, u_values, Rt, Rx, u_scale, k=5, model_k=None):
    """
    Build features for neural integrator (matching training format).
    
    Args:
        visited_pts: array of shape (N, 2) with (t, x) coordinates of visited points
        sampled: tuple (t, x) - the point to predict
        u_values: dictionary mapping (t, x) -> u value
        Rt: time radius for ellipse scaling
        Rx: space radius for ellipse scaling
        u_scale: normalization factor for u values
        k: number of neighbors to use for selection
        model_k: number of neighbors the model expects (if None, uses k)
    
    Returns:
        features: array of shape (model_k*3,) with [dt_scaled, dx_scaled, u_scaled] per neighbor
        neighbors: array of shape (k, 2) with neighbor coordinates used
    """
    if model_k is None:
        model_k = k
    
    # Get nearest k neighbors using elliptical metric
    neigh = nearest5_elliptical(visited_pts, sampled, Rt, Rx, k=k)
    
    # Ensure we have at least k neighbors (pad if needed)
    if len(neigh) < k:
        # Pad with last neighbor if not enough (shouldn't happen in practice)
        pad_neigh = np.zeros((k, 2), dtype=int)
        pad_neigh[:len(neigh)] = neigh
        if len(neigh) > 0:
            pad_neigh[len(neigh):] = neigh[-1]  # repeat last neighbor
        neigh = pad_neigh
    elif len(neigh) > k:
        neigh = neigh[:k]  # take only first k
    
    # Build features exactly as in training
    feats = []
    for (tn, xn) in neigh:
        # Relative indices scaled by ellipse radii (matching training: t0 - tn)
        dt_scaled = (sampled[0] - tn) / Rt
        dx_scaled = (sampled[1] - xn) / Rx
        u_scaled = u_values.get((int(tn), int(xn)), 0.0) / u_scale
        feats.append([dt_scaled, dx_scaled, u_scaled])
    
    # Pad or truncate to model_k if different from k
    if model_k != k:
        if model_k > k:
            # Pad with zeros (or repeat last neighbor)
            pad_feats = [[0.0, 0.0, 0.0]] * (model_k - k)
            feats.extend(pad_feats)
        else:
            # Truncate to model_k
            feats = feats[:model_k]
    
    feats = np.array(feats, dtype=np.float32).reshape(-1)  # (model_k*3,)
    return feats, neigh[:k]  # return only the k neighbors we actually used


In [21]:
def rollout_with_model(model, ground_truth_data, R=None, k=None, u_scale=None,
                      max_iterations=1000, device="cpu", seed=None, model_config=None):
    """Sequential integration rollout using trained neural integrator with nearest-k neighbors."""
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    
    Nt, Nx = ground_truth_data.shape
    is_bc = build_bc_mask(Nt, Nx)
    
    # Get parameters
    if model_config:
        model_k = model_config.get('k', 5)
        R = R or model_config.get('R', 10)
        k = k or model_k
        u_scale = u_scale or model_config.get('u_scale', np.max(np.abs(ground_truth_data)) + 1e-8)
    else:
        model_k, R, k = 5, R or 10, k or 5
        u_scale = u_scale or np.max(np.abs(ground_truth_data)) + 1e-8
    
    Rt, Rx = R, R * (Nx / Nt)
    
    # Initialize
    visited = np.zeros((Nt, Nx), dtype=bool)
    u_values = {}
    for x in range(Nx):
        visited[0, x] = True
        u_values[(0, x)] = float(ground_truth_data[0, x])
    for t in range(Nt):
        visited[t, 0] = True
        u_values[(t, 0)] = float(ground_truth_data[t, 0])
        visited[t, Nx-1] = True
        u_values[(t, Nx-1)] = float(ground_truth_data[t, Nx-1])
    
    history = []
    model.eval()
    
    for iteration in range(max_iterations):
        cand, _ = candidates_from_interior_elliptical(visited, is_bc, R, Rt, Rx)
        if len(cand) == 0:
            break
        
        # Absorb BC
        for (t, x) in cand:
            if is_bc[t, x]:
                visited[t, x] = True
                u_values[(t, x)] = float(ground_truth_data[t, x])
        
        interior_cand = [(t, x) for (t, x) in cand if not is_bc[t, x] and not visited[t, x]]
        if len(interior_cand) == 0:
            break
        
        sampled = random.choice(interior_cand)
        visited_pts = np.argwhere(visited)
        if len(visited_pts) < k:
            continue
        
        feats, neighbors = build_features_for_model(visited_pts, sampled, u_values, Rt, Rx, u_scale, k=k, model_k=model_k)
        
        with torch.no_grad():
            pred = model(torch.from_numpy(feats).unsqueeze(0).to(device)).item() * u_scale
        
        visited[sampled[0], sampled[1]] = True
        u_values[sampled] = pred
        history.append({
            'iteration': iteration, 'sampled': sampled, 'neighbors': neighbors.tolist(),
            'prediction': pred, 'ground_truth': float(ground_truth_data[sampled[0], sampled[1]])
        })
        
        if (iteration + 1) % 100 == 0:
            print(f"Iteration {iteration+1}: {sampled}, pred={pred:.6f}, true={ground_truth_data[sampled[0], sampled[1]]:.6f}")
    
    # Metrics
    visited_count = np.sum(visited)
    mse_errors = [(u_values[(t, x)] - ground_truth_data[t, x])**2 
                  for (t, x), _ in u_values.items() if not is_bc[t, x] and t > 0]
    avg_mse = np.mean(mse_errors) if mse_errors else 0.0
    
    metrics = {
        'total_iterations': len(history), 'coverage': visited_count / (Nt * Nx),
        'mse': avg_mse, 'rmse': np.sqrt(avg_mse)
    }
    print(f"Rollout: {len(history)} iterations, {metrics['coverage']:.2%} coverage, MSE={avg_mse:.6e}")
    
    return visited, u_values, history, metrics


In [22]:
# Alternative: Simple interactive version using mplcursors (if available)
# This provides hover tooltips and can be extended for clicking
try:
    import mplcursors
    
    def visualize_rollout_interactive_simple(visited, u_values, history, ground_truth_data, R, config):
        """Simpler interactive version using mplcursors for tooltips."""
        Nt, Nx = visited.shape
        Rt = R
        Rx = R * (Nx / Nt)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        visited_pts = np.array(list(u_values.keys()))
        history_dict = {tuple(h['sampled']): h for h in history}
        
        ic_bc_pts = []
        pred_pts = []
        pred_vals = []
        
        for pt in visited_pts:
            pt_tuple = tuple(pt)
            if pt_tuple in history_dict:
                pred_pts.append(pt)
                pred_vals.append(u_values[pt_tuple])
            else:
                ic_bc_pts.append(pt)
        
        # Plot IC/BC
        if len(ic_bc_pts) > 0:
            ic_bc_arr = np.array(ic_bc_pts)
            ax.scatter(ic_bc_arr[:, 1], ic_bc_arr[:, 0], 
                      s=20, c='gray', alpha=0.5, label='IC/BC', zorder=1)
        
        # Plot predicted points
        if len(pred_pts) > 0:
            pred_pts_arr = np.array(pred_pts)
            pred_vals_arr = np.array(pred_vals)
            scatter = ax.scatter(pred_pts_arr[:, 1], pred_pts_arr[:, 0], 
                              c=pred_vals_arr, s=40, cmap='viridis', 
                              alpha=0.8, edgecolors='black', linewidth=0.5, 
                              label='Predicted', zorder=2, picker=True)
            plt.colorbar(scatter, ax=ax, label='Predicted u value')
            
            # Add cursor for hover tooltips
            cursor = mplcursors.cursor(scatter, hover=True)
            
            @cursor.connect("add")
            def on_add(sel):
                idx = sel.target.index
                pt = tuple(pred_pts_arr[idx])
                if pt in history_dict:
                    h = history_dict[pt]
                    neighbors = h['neighbors']
                    pred = h['prediction']
                    true = h['ground_truth']
                    error = pred - true
                    sel.annotation.set_text(
                        f"Point: {pt}\n"
                        f"Pred: {pred:.4f}\n"
                        f"True: {true:.4f}\n"
                        f"Error: {error:.4f}\n"
                        f"Neighbors: {len(neighbors)}"
                    )
        
        ax.set_xlabel('x (space)')
        ax.set_ylabel('t (time)')
        ax.set_title('Hover over predicted points to see details')
        ax.set_aspect('auto')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
    print("mplcursors available - use visualize_rollout_interactive_simple() for hover tooltips")
except ImportError:
    print("mplcursors not available. Install with: pip install mplcursors")
    print("Or use the main visualize_rollout_interactive() function with %matplotlib widget")


mplcursors available - use visualize_rollout_interactive_simple() for hover tooltips


In [23]:
# Simple non-interactive version that always displays
def visualize_rollout_simple(visited, u_values, history, ground_truth_data, R, config):
    """Simple visualization that always displays (non-interactive)."""
    Nt, Nx = visited.shape
    Rt = R
    Rx = R * (Nx / Nt)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    visited_pts = np.array(list(u_values.keys()))
    history_dict = {tuple(h['sampled']): h for h in history}
    
    ic_bc_pts = []
    pred_pts = []
    pred_vals = []
    
    for pt in visited_pts:
        pt_tuple = tuple(pt)
        if pt_tuple in history_dict:
            pred_pts.append(pt)
            pred_vals.append(u_values[pt_tuple])
        else:
            ic_bc_pts.append(pt)
    
    print(f"Plotting: {len(ic_bc_pts)} IC/BC points, {len(pred_pts)} predicted points")
    
    # Plot IC/BC
    if len(ic_bc_pts) > 0:
        ic_bc_arr = np.array(ic_bc_pts)
        ax.scatter(ic_bc_arr[:, 1], ic_bc_arr[:, 0], 
                  s=20, c='gray', alpha=0.5, label='IC/BC', zorder=1)
    
    # Plot predicted points
    if len(pred_pts) > 0:
        pred_pts_arr = np.array(pred_pts)
        pred_vals_arr = np.array(pred_vals)
        scatter = ax.scatter(pred_pts_arr[:, 1], pred_pts_arr[:, 0], 
                          c=pred_vals_arr, s=40, cmap='viridis', 
                          alpha=0.8, edgecolors='black', linewidth=0.5, 
                          label='Predicted', zorder=2)
        plt.colorbar(scatter, ax=ax, label='Predicted u value')
    else:
        ax.text(0.5, 0.5, 'No predicted points to display', 
               transform=ax.transAxes, ha='center', va='center', fontsize=14)
    
    ax.set_xlabel('x (space)')
    ax.set_ylabel('t (time)')
    ax.set_title('Rollout Results - Predicted Points')
    ax.set_aspect('auto')
    if len(pred_pts) > 0 or len(ic_bc_pts) > 0:
        ax.legend()
    plt.tight_layout()
    
    # Force display
    display(fig)  # Use IPython display for Jupyter
    print(f"\n✓ Displayed {len(pred_pts)} predicted points and {len(ic_bc_pts)} IC/BC points")
    
    return fig, ax


In [24]:
# Interactive visualization with clickable points
def visualize_rollout_interactive(visited, u_values, history, ground_truth_data, R, config):
    """Interactive visualization: click on points to see neighbors and predictions."""
    Nt, Nx = visited.shape
    Rt = R
    Rx = R * (Nx / Nt)
    is_bc = build_bc_mask(Nt, Nx)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Separate IC/BC from predicted points
    visited_pts = np.array(list(u_values.keys()))
    history_dict = {tuple(h['sampled']): h for h in history}
    
    ic_bc_pts = []
    pred_pts = []
    pred_vals = []
    
    for pt in visited_pts:
        pt_tuple = tuple(pt)
        if pt_tuple in history_dict:
            pred_pts.append(pt)
            pred_vals.append(u_values[pt_tuple])
        else:
            ic_bc_pts.append(pt)
    
    # Plot IC/BC points in gray
    if len(ic_bc_pts) > 0:
        ic_bc_arr = np.array(ic_bc_pts)
        ax.scatter(ic_bc_arr[:, 1], ic_bc_arr[:, 0], 
                  s=20, c='gray', alpha=0.5, label='IC/BC', zorder=1)
    
    # Plot predicted points colored by value
    if len(pred_pts) > 0:
        pred_pts_arr = np.array(pred_pts)
        pred_vals_arr = np.array(pred_vals)
        scatter = ax.scatter(pred_pts_arr[:, 1], pred_pts_arr[:, 0], 
                          c=pred_vals_arr, s=40, cmap='viridis', 
                          alpha=0.8, edgecolors='black', linewidth=0.5, 
                          label='Predicted', zorder=2, picker=True)
        plt.colorbar(scatter, ax=ax, label='Predicted u value')
    else:
        pred_pts_arr = np.array([])
        pred_vals_arr = np.array([])
        ax.text(0.5, 0.5, 'No predicted points to display', 
               transform=ax.transAxes, ha='center', va='center', fontsize=14)
    
    ax.set_xlabel('x (space)')
    ax.set_ylabel('t (time)')
    ax.set_title('Click on a predicted point (colored) to see its neighbors and prediction details')
    ax.set_aspect('auto')
    if len(pred_pts) > 0 or len(ic_bc_pts) > 0:
        ax.legend()
    
    # Store original plot elements for click handler
    original_pts = pred_pts_arr if len(pred_pts) > 0 else np.array([])
    
    def on_click(event):
        if event.inaxes != ax or len(pred_pts) == 0:
            return
        
        click_x, click_t = event.xdata, event.ydata
        if click_x is None or click_t is None:
            return
        
        # Find closest predicted point
        distances = np.sqrt((pred_pts_arr[:, 1] - click_x)**2 + (pred_pts_arr[:, 0] - click_t)**2)
        nearest_idx = np.argmin(distances)
        nearest_pt = tuple(pred_pts_arr[nearest_idx])
        
        # Check if close enough (within reasonable click distance)
        if distances[nearest_idx] > 2.0:  # threshold for click detection
            return
        
        if nearest_pt not in history_dict:
            return
        
        h = history_dict[nearest_pt]
        neighbors = np.array(h['neighbors'])
        pred = h['prediction']
        true = h['ground_truth']
        error = pred - true
        
        # Clear and replot
        ax.clear()
        
        # Replot IC/BC
        if len(ic_bc_pts) > 0:
            ic_bc_arr = np.array(ic_bc_pts)
            ax.scatter(ic_bc_arr[:, 1], ic_bc_arr[:, 0], 
                      s=15, c='gray', alpha=0.3, zorder=1)
        
        # Replot all predicted points (faded)
        scatter = ax.scatter(pred_pts_arr[:, 1], pred_pts_arr[:, 0], 
                          c=pred_vals_arr, s=25, cmap='viridis', 
                          alpha=0.3, edgecolors='black', linewidth=0.2, zorder=2)
        plt.colorbar(scatter, ax=ax, label='Predicted u value')
        
        # Highlight selected point
        ax.scatter([nearest_pt[1]], [nearest_pt[0]], 
                  c='red', s=300, alpha=0.9, edgecolors='black', 
                  linewidth=3, marker='*', label='Selected', zorder=10)
        
        # Show neighbors
        if len(neighbors) > 0:
            ax.scatter(neighbors[:, 1], neighbors[:, 0], 
                      c='blue', s=150, alpha=0.9, edgecolors='black', 
                      linewidth=2, marker='o', label='Neighbors', zorder=9)
            # Draw connections
            for (tn, xn) in neighbors:
                ax.plot([nearest_pt[1], xn], [nearest_pt[0], tn], 
                       'cyan', alpha=0.7, linewidth=2.5, zorder=8)
        
        # Draw ellipse around selected point
        ellipse = patches.Ellipse((nearest_pt[1], nearest_pt[0]), 
                                 width=2*Rx, height=2*Rt,
                                 fill=False, color='purple', 
                                 alpha=0.4, linewidth=2, linestyle='--', zorder=7)
        ax.add_patch(ellipse)
        
        ax.set_xlabel('x (space)')
        ax.set_ylabel('t (time)')
        ax.set_title(f'Point {nearest_pt}: pred={pred:.6f}, true={true:.6f}, error={error:.6f}')
        ax.legend(loc='upper right', fontsize=9)
        ax.set_aspect('auto')
        
        plt.draw()
        print(f"\n{'='*60}")
        print(f"Point: {nearest_pt}")
        print(f"  Predicted: {pred:.6f}")
        print(f"  Ground Truth: {true:.6f}")
        print(f"  Error: {error:.6f} ({abs(error):.2%} of true value)")
        print(f"  Neighbors: {neighbors.tolist()}")
        print(f"{'='*60}")
    
    # Connect the click event handler
    cid = fig.canvas.mpl_connect('button_press_event', on_click)
    
    # Store connection ID and data in figure for persistence
    fig._click_handler = cid
    fig._pred_pts_arr = pred_pts_arr if len(pred_pts) > 0 else np.array([])
    fig._history_dict = history_dict
    fig._ic_bc_pts = ic_bc_pts
    fig._u_values = u_values
    fig._Rt = Rt
    fig._Rx = Rx
    
    plt.tight_layout()
    
    # Display the initial plot
    plt.show()
    
    print(f"\n✓ Plot displayed: {len(pred_pts)} predicted points, {len(ic_bc_pts)} IC/BC points")
    print("✓ Interactive mode enabled. Click on any colored (predicted) point to see its neighbors!")
    print("  Note: If clicking doesn't work, make sure you ran cell 1 (%matplotlib widget)")
    
    return fig, ax  # Return figure so it stays alive

# Visualization
def visualize_rollout(visited, u_values, ground_truth_data, show_ground_truth=False, zoom_y=None, zoom_window=50):
    """Visualize rollout results.
    
    Args:
        visited: boolean array (Nt, Nx)
        u_values: dict mapping (t, x) -> u value
        ground_truth_data: array (Nt, Nx)
        show_ground_truth: whether to show comparison plots
        zoom_y: y-coordinate (time step) to zoom to, or None for full view
        zoom_window: window size around zoom_y (only used if zoom_y is not None)
    """
    import numpy.ma as ma
    Nt, Nx = visited.shape
    pred_array = np.full((Nt, Nx), np.nan)
    for (t, x), val in u_values.items():
        pred_array[t, x] = val
    
    # Determine zoom region
    if zoom_y is not None:
        y_min = max(0, int(zoom_y - zoom_window))
        y_max = min(Nt, int(zoom_y + zoom_window))
        x_min, x_max = 0, Nx
        pred_array = pred_array[y_min:y_max, x_min:x_max]
        ground_truth_data = ground_truth_data[y_min:y_max, x_min:x_max]
        visited_zoom = visited[y_min:y_max, x_min:x_max]
    else:
        visited_zoom = visited
    
    if show_ground_truth:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, arr, title in zip(axes, [pred_array, ground_truth_data, pred_array - ground_truth_data],
                                 ['Predicted', 'Ground Truth', 'Error']):
            # Mask NaN values for better display
            masked_arr = ma.masked_invalid(arr)
            vmin, vmax = np.nanmin(arr), np.nanmax(arr)
            if np.isnan(vmin) or np.isnan(vmax):
                vmin, vmax = -1, 1  # fallback
            
            im = ax.imshow(masked_arr, aspect='auto', origin='lower', 
                          cmap='viridis' if title != 'Error' else 'RdBu',
                          vmin=vmin, vmax=vmax)
            ax.set_title(title + (f' (zoomed to y={zoom_y})' if zoom_y is not None else ''))
            ax.set_xlabel('x (space)')
            ax.set_ylabel('t (time)')
            plt.colorbar(im, ax=ax)
            
            # Overlay visited points as scatter
            if title == 'Predicted':
                visited_pts = np.argwhere(visited_zoom)
                if len(visited_pts) > 0:
                    ax.scatter(visited_pts[:, 1], visited_pts[:, 0], 
                             c='red', s=1, alpha=0.3, marker='.', label='visited')
    else:
        masked_arr = ma.masked_invalid(pred_array)
        vmin, vmax = np.nanmin(pred_array), np.nanmax(pred_array)
        if np.isnan(vmin) or np.isnan(vmax):
            vmin, vmax = -1, 1
        plt.imshow(masked_arr, aspect='auto', origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        plt.colorbar()
        plt.title('Predicted u values' + (f' (zoomed to y={zoom_y})' if zoom_y is not None else ''))
        plt.xlabel('x (space)')
        plt.ylabel('t (time)')
        # Overlay visited points
        visited_pts = np.argwhere(visited_zoom)
        if len(visited_pts) > 0:
            plt.scatter(visited_pts[:, 1], visited_pts[:, 0], 
                       c='red', s=1, alpha=0.3, marker='.')
    plt.tight_layout()
    plt.show()


In [25]:
# -----------------------
# 6. Ablation study framework
# -----------------------

def run_ablation_study(model_path, ground_truth_data, 
                      k_values=[3, 5, 7, 10],
                      R_values=[5, 10, 15, 20],
                      max_iterations=1000,
                      device="cpu",
                      save_results=True,
                      results_dir="ablation_results"):
    """
    Run ablation study varying k and R values.
    
    Returns:
        results: dictionary with all experiment results
    """
    if save_results and not os.path.exists(results_dir):
        os.makedirs(results_dir)
    
    results = {}
    
    # Load base model
    base_model, base_config = load_integrator_model(model_path, device=device)
    base_u_scale = base_config['u_scale']
    
    total_experiments = len(k_values) * len(R_values)
    exp_count = 0
    
    for k in k_values:
        for R in R_values:
            exp_count += 1
            exp_name = f"k{k}_R{R}"
            print(f"\n{'='*60}")
            print(f"Experiment {exp_count}/{total_experiments}: {exp_name}")
            print(f"{'='*60}")
            
            # Create model with new k (if different)
            if k != base_config['k']:
                # Need to retrain or use different model architecture
                # For now, we'll use the base model but with different k in feature building
                # This is a limitation - ideally we'd have models trained for each k
                print(f"Warning: Using base model (k={base_config['k']}) with k={k} in features")
                model = base_model
            else:
                model = base_model
            
            start_time = time.time()
            
            # Run rollout
            visited, u_values, history, metrics = rollout_with_model(
                model, ground_truth_data, R=R, k=k, u_scale=base_u_scale,
                max_iterations=max_iterations, device=device, seed=42
            )
            
            elapsed_time = time.time() - start_time
            
            # Store results
            results[exp_name] = {
                'k': k,
                'R': R,
                'metrics': metrics,
                'elapsed_time': elapsed_time,
                'history_length': len(history)
            }
            
            print(f"Completed in {elapsed_time:.2f}s")
    
    # Summary
    print(f"\n{'='*60}")
    print("ABLATION STUDY SUMMARY")
    print(f"{'='*60}")
    
    # Create summary table
    print(f"\n{'k':<5} {'R':<5} {'Coverage':<12} {'MSE':<15} {'RMSE':<15} {'Time (s)':<10}")
    print("-" * 70)
    
    for exp_name, exp_results in results.items():
        k = exp_results['k']
        R = exp_results['R']
        metrics = exp_results['metrics']
        print(f"{k:<5} {R:<5} {metrics['coverage']:<12.2%} "
              f"{metrics['mse']:<15.6e} {metrics['rmse']:<15.6e} "
              f"{exp_results['elapsed_time']:<10.2f}")
    
    # Save results
    if save_results:
        results_file = os.path.join(results_dir, "ablation_results.json")
        # Convert numpy types to Python types for JSON
        results_json = {}
        for exp_name, exp_data in results.items():
            results_json[exp_name] = {
                'k': int(exp_data['k']),
                'R': int(exp_data['R']),
                'metrics': {
                    'total_iterations': int(exp_data['metrics']['total_iterations']),
                    'coverage': float(exp_data['metrics']['coverage']),
                    'visited_count': int(exp_data['metrics']['visited_count']),
                    'total_points': int(exp_data['metrics']['total_points']),
                    'mse': float(exp_data['metrics']['mse']),
                    'rmse': float(exp_data['metrics']['rmse'])
                },
                'elapsed_time': float(exp_data['elapsed_time']),
                'history_length': int(exp_data['history_length'])
            }
        
        with open(results_file, 'w') as f:
            json.dump(results_json, f, indent=2)
        print(f"\nResults saved to {results_file}")
    
    return results


def plot_ablation_results(results, save_path=None):
    """
    Plot ablation study results.
    """
    # Extract data
    k_vals = sorted(set(r['k'] for r in results.values()))
    R_vals = sorted(set(r['R'] for r in results.values()))
    
    # Create coverage and MSE matrices
    coverage_matrix = np.zeros((len(k_vals), len(R_vals)))
    mse_matrix = np.zeros((len(k_vals), len(R_vals)))
    
    for exp_name, exp_data in results.items():
        k = exp_data['k']
        R = exp_data['R']
        k_idx = k_vals.index(k)
        R_idx = R_vals.index(R)
        coverage_matrix[k_idx, R_idx] = exp_data['metrics']['coverage']
        mse_matrix[k_idx, R_idx] = exp_data['metrics']['mse']
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Coverage heatmap
    im1 = axes[0].imshow(coverage_matrix, aspect='auto', cmap='viridis', origin='lower')
    axes[0].set_title('Coverage by k and R')
    axes[0].set_xlabel('R')
    axes[0].set_ylabel('k')
    axes[0].set_xticks(range(len(R_vals)))
    axes[0].set_xticklabels(R_vals)
    axes[0].set_yticks(range(len(k_vals)))
    axes[0].set_yticklabels(k_vals)
    plt.colorbar(im1, ax=axes[0])
    
    # MSE heatmap
    im2 = axes[1].imshow(mse_matrix, aspect='auto', cmap='Reds', origin='lower')
    axes[1].set_title('MSE by k and R')
    axes[1].set_xlabel('R')
    axes[1].set_ylabel('k')
    axes[1].set_xticks(range(len(R_vals)))
    axes[1].set_xticklabels(R_vals)
    axes[1].set_yticks(range(len(k_vals)))
    axes[1].set_yticklabels(k_vals)
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.show()


In [26]:
# Error propagation analysis over time
def visualize_error_propagation(visited, u_values, history, ground_truth_data):
    """
    Visualize how prediction errors propagate over time.
    
    Shows:
    1. Error vs time (scatter and statistics)
    2. Error distribution over time
    3. Cumulative error statistics
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Extract errors from history
    errors_by_time = {}
    abs_errors_by_time = {}
    
    for h in history:
        t, x = h['sampled']
        pred = h['prediction']
        true = h['ground_truth']
        error = pred - true
        abs_error = abs(error)
        
        if t not in errors_by_time:
            errors_by_time[t] = []
            abs_errors_by_time[t] = []
        
        errors_by_time[t].append(error)
        abs_errors_by_time[t].append(abs_error)
    
    if len(errors_by_time) == 0:
        print("No predictions found in history!")
        return None, None
    
    # Prepare data for plotting
    times = sorted(errors_by_time.keys())
    mean_errors = [np.mean(errors_by_time[t]) for t in times]
    std_errors = [np.std(errors_by_time[t]) for t in times]
    mean_abs_errors = [np.mean(abs_errors_by_time[t]) for t in times]
    max_abs_errors = [np.max(abs_errors_by_time[t]) for t in times]
    min_abs_errors = [np.min(abs_errors_by_time[t]) for t in times]
    counts = [len(errors_by_time[t]) for t in times]
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Mean error vs time (with std bands)
    ax1 = axes[0, 0]
    ax1.plot(times, mean_errors, 'b-', linewidth=2, label='Mean Error', marker='o', markersize=4)
    ax1.fill_between(times, 
                     [m - s for m, s in zip(mean_errors, std_errors)],
                     [m + s for m, s in zip(mean_errors, std_errors)],
                     alpha=0.3, color='blue', label='±1 Std')
    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax1.set_xlabel('Time (t)')
    ax1.set_ylabel('Error (Predicted - True)')
    ax1.set_title('Mean Error Propagation Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Absolute error statistics vs time
    ax2 = axes[0, 1]
    ax2.plot(times, mean_abs_errors, 'r-', linewidth=2, label='Mean |Error|', marker='o', markersize=4)
    ax2.plot(times, max_abs_errors, 'r--', linewidth=1.5, label='Max |Error|', alpha=0.7)
    ax2.plot(times, min_abs_errors, 'r:', linewidth=1.5, label='Min |Error|', alpha=0.7)
    ax2.fill_between(times, min_abs_errors, max_abs_errors, alpha=0.2, color='red')
    ax2.set_xlabel('Time (t)')
    ax2.set_ylabel('Absolute Error')
    ax2.set_title('Absolute Error Statistics Over Time')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')  # Log scale for better visualization
    
    # 3. Number of predictions per time step
    ax3 = axes[1, 0]
    ax3.bar(times, counts, alpha=0.6, color='green', edgecolor='black', linewidth=0.5)
    ax3.set_xlabel('Time (t)')
    ax3.set_ylabel('Number of Predictions')
    ax3.set_title('Prediction Density Over Time')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Error distribution heatmap (time vs error magnitude)
    ax4 = axes[1, 1]
    # Create bins for error magnitude
    all_abs_errors = [abs_err for t in times for abs_err in abs_errors_by_time[t]]
    if len(all_abs_errors) > 0:
        error_bins = np.logspace(np.log10(min(all_abs_errors) + 1e-10), 
                                 np.log10(max(all_abs_errors) + 1e-10), 20)
        error_hist = np.zeros((len(times), len(error_bins)-1))
        
        for i, t in enumerate(times):
            hist, _ = np.histogram(abs_errors_by_time[t], bins=error_bins)
            error_hist[i, :] = hist
        
        im = ax4.imshow(error_hist.T, aspect='auto', origin='lower', 
                       cmap='YlOrRd', interpolation='nearest')
        ax4.set_xlabel('Time (t)')
        ax4.set_ylabel('Error Magnitude Bin (log scale)')
        ax4.set_title('Error Distribution Over Time')
        ax4.set_xticks(range(0, len(times), max(1, len(times)//10)))
        ax4.set_xticklabels([times[i] for i in range(0, len(times), max(1, len(times)//10))])
        plt.colorbar(im, ax=ax4, label='Count')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print("ERROR PROPAGATION SUMMARY")
    print(f"{'='*60}")
    print(f"Total time steps with predictions: {len(times)}")
    print(f"Time range: t={min(times)} to t={max(times)}")
    print(f"\nOverall Statistics:")
    all_errors = [e for t in times for e in errors_by_time[t]]
    all_abs_errors_flat = [abs(e) for e in all_errors]
    print(f"  Mean error: {np.mean(all_errors):.6f}")
    print(f"  Std error: {np.std(all_errors):.6f}")
    print(f"  Mean |error|: {np.mean(all_abs_errors_flat):.6f}")
    print(f"  Max |error|: {np.max(all_abs_errors_flat):.6f}")
    print(f"  RMSE: {np.sqrt(np.mean([e**2 for e in all_errors])):.6f}")
    
    # Error trend analysis
    if len(times) > 1:
        # Linear fit to mean absolute error
        coeffs = np.polyfit(times, mean_abs_errors, 1)
        trend = "increasing" if coeffs[0] > 0 else "decreasing"
        print(f"\nError Trend:")
        print(f"  Mean |error| trend: {trend} (slope: {coeffs[0]:.2e} per time step)")
        print(f"  Early time (first 10%): mean |error| = {np.mean(mean_abs_errors[:max(1, len(times)//10)]):.6f}")
        print(f"  Late time (last 10%): mean |error| = {np.mean(mean_abs_errors[-max(1, len(times)//10):]):.6f}")
    
    print(f"{'='*60}\n")
    
    return fig, axes


In [27]:
# Generate ground truth data (same as training)
def diffusion_solver_dirichlet(u0, nu=0.01, dx=1/128, dt=1e-4, n_steps=2000):
    u = u0.copy()
    Nx = len(u)
    traj = [u.copy()]
    for _ in range(n_steps):
        un = u.copy()
        lap = (un[2:] - 2*un[1:-1] + un[:-2]) / dx**2
        u[1:-1] = un[1:-1] + nu * dt * lap
        u[0] = 0.0
        u[-1] = 0.0
        traj.append(u.copy())
    return np.stack(traj)

def generate_diffusion_data(Nx=500, nu=0.01, dt=1e-4, n_steps=2000):
    dx = 1.0 / (Nx - 1)
    x = np.linspace(0, 1, Nx)
    u0 = np.sin(2*np.pi * x)
    data = diffusion_solver_dirichlet(u0, nu, dx, dt, n_steps)
    return x, data

x, data = generate_diffusion_data(Nx=500, nu=0.01, dt=1e-4, n_steps=2000)
print(f"Data shape: {data.shape}")


Data shape: (2001, 500)


In [28]:
# Run rollout
model, config = load_integrator_model("model_integrator.pth", device="cuda")
visited, u_values, history, metrics = rollout_with_model(
    model, data, model_config=config, max_iterations=10000, device="cuda", seed=42
) 

Model loaded: k=5, R=10, u_scale=9.999951e-01
Iteration 100: (6, 65), pred=0.719660, true=0.729915
Iteration 200: (15, 389), pred=-0.962716, true=-0.982221
Iteration 300: (18, 317), pred=-0.745202, true=-0.750700
Iteration 400: (9, 101), pred=0.969035, true=0.955278
Iteration 500: (18, 300), pred=-0.576881, true=-0.593459
Iteration 600: (15, 114), pred=0.966619, true=0.990266
Iteration 700: (1, 221), pred=0.335081, true=0.351193
Iteration 800: (25, 93), pred=0.961064, true=0.920237
Iteration 900: (13, 308), pred=-0.669289, true=-0.671433
Iteration 1000: (41, 249), pred=-0.088056, true=0.006286
Rollout: 1000 iterations, 0.55% coverage, MSE=5.508822e-04


In [29]:
# Interactive visualization (click on points to see neighbors and predictions)
# This shows all visited points colored by predicted value
# Click on any point to see its 5 nearest neighbors and prediction details
# 
# IMPORTANT: Make sure cell 1 (%matplotlib widget) was run first!

if 'visited' in globals() and 'u_values' in globals():
    fig, ax = visualize_rollout_interactive(visited, u_values, history, data, config['R'], config)
else:
    print("⚠️  Please run cell 13 (rollout) first!")


<IPython.core.display.Javascript object>


✓ Plot displayed: 1000 predicted points, 4500 IC/BC points
✓ Interactive mode enabled. Click on any colored (predicted) point to see its neighbors!
  Note: If clicking doesn't work, make sure you ran cell 1 (%matplotlib widget)


In [30]:
# Error propagation analysis over time
# This shows how prediction errors evolve as the rollout progresses
if 'visited' in globals() and 'u_values' in globals() and 'history' in globals():
    print("Analyzing error propagation over time...")
    fig_err, axes_err = visualize_error_propagation(visited, u_values, history, data)
else:
    print("⚠️  Please run cell 13 (rollout) first!")


Analyzing error propagation over time...


<IPython.core.display.Javascript object>


ERROR PROPAGATION SUMMARY
Total time steps with predictions: 49
Time range: t=1 to t=55

Overall Statistics:
  Mean error: 0.003147
  Std error: 0.023259
  Mean |error|: 0.015974
  Max |error|: 0.135035
  RMSE: 0.023471

Error Trend:
  Mean |error| trend: increasing (slope: 1.06e-03 per time step)
  Early time (first 10%): mean |error| = 0.007250
  Late time (last 10%): mean |error| = 0.044341

